In [ ]:

# ==========================
# 
# Libraries
#
# ==========================


# 0) Imports
import time  # timing hyperparameter searches
from collections import defaultdict  # convenient dict for collecting results

import numpy as np  # numerical utilities
import pandas as pd  # dataframes + IO
import matplotlib.pyplot as plt

from sklearn.model_selection import train_test_split, StratifiedKFold, cross_validate  # split + CV tools
from sklearn.preprocessing import StandardScaler, OneHotEncoder  # feature scaling
from sklearn.compose import ColumnTransformer  # column-wise preprocessing
from sklearn.pipeline import Pipeline as SkPipeline  # sklearn pipeline for preprocess steps
from imblearn.pipeline import Pipeline as ImbPipeline  # imblearn pipeline (supports resampling)
from imblearn.over_sampling import SMOTE  # minority class oversampling
from sklearn.base import clone # for copying
from imblearn.under_sampling import RandomUnderSampler

from sklearn.impute import SimpleImputer  # fill missing values
from sklearn.metrics import classification_report, confusion_matrix, accuracy_score, roc_auc_score, precision_recall_curve, precision_score, recall_score, make_scorer, f1_score, roc_curve # evaluation metrics
from sklearn.linear_model import LogisticRegression  # LR baseline
from sklearn.tree import DecisionTreeClassifier  # decision tree
from sklearn.ensemble import RandomForestClassifier  # random forest
from sklearn.svm import LinearSVC, SVC  # linear SVM + RBF SVM
from sklearn.neural_network import MLPClassifier  # shallow neural net
from sklearn.calibration import CalibratedClassifierCV  # probability calibration wrapper
from xgboost import XGBClassifier  # gradient boosting (XGBoost)

from skopt import BayesSearchCV  # Bayesian hyperparameter search
from skopt.space import Real, Categorical, Integer  # search spaces
from scipy import stats

# ==========================
# 
# Data
#
# ==========================

# 1) Global config
RANDOM_STATE = 29  # fixed seed for reproducibility

# 2) Data loader
def load_data(orig_path, perf_path):  # load origination + performance files and create labels
    orig_cols = [  # expected origination columns
        "credit_score","first_payment_date","first_time_homebuyer_flag","maturity_date",
        "Metropolitan Statistical Area (msa)","mortgage_insurance_percent","number_of_units",
        "occupancy_status","original_combined_loan_to_value","original_debt_to_income_ratio",
        "original_upb","original_loan_to_value","original_interest_rate","channel",
        "prepayment_penalty_flag","product_type","property_state","property_type","postal_code",
        "loan_id","loan_purpose","original_loan_term","number_of_borrowers","seller_name",
        "servicer_name","super_conforming_flag","pre_harp_loan_id","program_indicator",
        "harp_indicator","property_valuation_method","interest_only_indicator",
        "Mortgage Insurance Cancellation"
    ]
    perf_cols = [  # expected performance columns
        "loan_id","monthly_reporting_period","current_actual_upb","current_loan_delinquency_status",
        "loan_age","remaining_months_to_legal_maturity","defect_settlement_date","mod_flag",
        "zero_balance_code","zero_balance_effective_date","interest_rate","current_deferred_upb",
        "last_paid_installment_date","mi_recoveries","net_sales_proceeds","non_mi_recoveries",
        "expenses","legal_costs","maintenance_and_preservation_costs","taxes_and_insurance",
        "miscellaneous_expenses","actual_loss","modification_cost","step_mod_flag",
        "deferred_payment_plan","estimated_ltv","zero_balance_removal_upb",
        "delinquent_accrued_interest","delinquency_due_to_disaster",
        "borrower_assistance_status_code","current_month_modification_cost","interest_bearing_upb"
    ]
    df_orig = pd.read_csv(orig_path, sep='|', names=orig_cols, header=None, low_memory=False)  # read origination
    df_per  = pd.read_csv(perf_path,  sep='|', names=perf_cols, header=None, low_memory=False)  # read performance

    df_per['current_loan_delinquency_status'] = pd.to_numeric(  # delinquency to numeric
        df_per['current_loan_delinquency_status'], errors='coerce'
    )
    default_ids = df_per[df_per['current_loan_delinquency_status'] >= 1]['loan_id'].unique()  # loans ever 30+ DPD

    df = df_orig.merge(pd.DataFrame({'loan_id': default_ids, 'default_flag': 1}),  # mark defaults via merge
                       on='loan_id', how='left')
    df['default_flag'] = df['default_flag'].fillna(0).astype(int)  # non-defaults -> 0

    df['first_time_homebuyer_flag'] = (  # normalise Y/N to 1/0
        df['first_time_homebuyer_flag'].astype(str).str.strip().str.upper()
        .map({'Y': 1, 'N': 0}).fillna(0).astype(int)
    )
    return df  # labelled origination rows

# 3) Load data  
df = load_data("sample_orig_2019.txt", "sample_svcg_2019.txt")  #  file paths for this run

# 4) Features/target, split
user_features = [  # the 10 predictors for Exp1
    'credit_score','first_time_homebuyer_flag','mortgage_insurance_percent',
    'original_combined_loan_to_value','original_debt_to_income_ratio','original_upb',
    'original_loan_to_value','original_interest_rate','original_loan_term','number_of_borrowers'
]
X = df[user_features].copy()  # feature matrix
y = df['default_flag'].astype(int).copy()  # binary target

X_train, X_test, y_train, y_test = train_test_split(  # 80/20 stratified split
    X, y, test_size=0.2, stratify=y, random_state=RANDOM_STATE
)

# 5) Preprocessing:
# - median imputation for robustness
# - standard scaling 
# - ColumnTransformer applies the numeric pipeline only to the selected columns
num_pipe = SkPipeline(steps=[
    ("imp", SimpleImputer(strategy="median")),  # fills missing numeric values
    ("sc", StandardScaler())  # standardise features
])
preprocess = ColumnTransformer(
    transformers=[("num", num_pipe, user_features)],  # apply to numeric feature list
    remainder="drop"  # drop anything else
)

# ==========================
# 
# Experiment 1 
#
# ==========================


# 6)  pipeline builder (preprocess + SMOTE + model)
def make_pipe(clf):  # wraps a classifier with preprocessing and resampling
    return ImbPipeline(steps=[
        ("prep", preprocess),  # impute + scale 
        ("smote", SMOTE(random_state=RANDOM_STATE)),  # oversample inside each CV fold
        ("clf", clf)  # the classifier to tune
    ])

# 7) CV + scoring 
cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=RANDOM_STATE)  # 5-fold stratified CV
scoring = {  # metrics to compute during cross_validate
    "accuracy": make_scorer(accuracy_score),
    "precision": make_scorer(precision_score, zero_division=0),
    "recall": make_scorer(recall_score, zero_division=0),
    "f1": make_scorer(f1_score, zero_division=0),
    "roc_auc": "roc_auc"
}

def cv_metrics_formatted(estimator, X, y, cv):  # run CV 
    scores = cross_validate(estimator, X, y, scoring=scoring, cv=cv, n_jobs=-1, return_train_score=False)  # CV run
    summary = {}  # collect metrics here
    for metric in scoring.keys():  # for each metric requested
        mean = np.mean(scores[f"test_{metric}"])  # mean across folds
        std  = np.std(scores[f"test_{metric}"])  # std across folds
        summary[metric] = f"{mean:.4f} ± {std:.4f}"  # formatted string
    return summary  # dict: metric -> "mean ± std"

# 8) Threshold policy 
def choose_threshold(y_true, y_proba):  # pick the threshold that maximises F1
    prec, rec, thr = precision_recall_curve(y_true, y_proba)  # precision/recall across thresholds
    f1s = (2 * prec * rec) / (prec + rec + 1e-9)  # compute F1 for each point
    idx = int(np.nanargmax(f1s))  # index of best F1
    return float(thr[max(idx - 1, 0)]) if len(thr) else 0.5  # map to threshold array or default 0.5

# 9) Bayesian search wrapper
def fit_search(pipe, space, Xtr, ytr, n_iter=20):  # hyperparameter search via Bayes optimisation
    search = BayesSearchCV(
        estimator=pipe,  # full pipeline (prep + SMOTE + model)
        search_spaces=space,  # hyperparameter dictionary for this model
        n_iter=n_iter,  # number of Bayes iterations
        scoring="roc_auc",  # objective to maximise
        cv=cv,  # 5-fold CV
        n_jobs=-1,  
        random_state=RANDOM_STATE,  # reproducibility
        verbose=0  
    )
    t0 = time.time()  # start timer
    search.fit(Xtr, ytr)  # run the optimisation on the training split
    return search, time.time() - t0  # return fitted search 

# 10) Model registry (pipelines + search spaces)
models = {

#    "SVC_RBF": (  # kernel SVM with RBF kernel (slowest; you kept probability=True)
#        make_pipe(SVC(kernel="rbf", probability=True, gamma="scale", cache_size=2000, random_state=RANDOM_STATE)),
#        {
#            "clf__C": Real(0.10, 100.00, prior="log-uniform"),  # tune soft-margin C
#            # "clf__gamma": Real(0.0001, 0.1, prior="log-uniform")  # optional if not fixing gamma="scale"
#        }
#    ),

    "LogisticRegression": (  # LR with scaling
        make_pipe(LogisticRegression(max_iter=500, random_state=RANDOM_STATE)),
        {
            "clf__C": Real(0.001, 100.0, prior="log-uniform"),
            "clf__solver": Categorical(["lbfgs","saga"])
        }
    ),
    "DecisionTree": (  # single tree
        make_pipe(DecisionTreeClassifier(random_state=RANDOM_STATE)),
        {
            "clf__max_depth": Integer(3, 12),
            "clf__min_samples_split": Integer(2, 20),
            "clf__min_samples_leaf": Integer(1, 10)
        }
    ),
    "RandomForest": (  # ensemble of trees
        make_pipe(RandomForestClassifier(random_state=RANDOM_STATE, n_jobs=-1)),
        {
            "clf__n_estimators": Integer(100, 400),
            "clf__max_depth": Integer(4, 18),
            "clf__min_samples_split": Integer(2, 20)
        }
    ),
    "XGBoost": (  # gradient boosted trees (hist for speed)
        make_pipe(XGBClassifier(eval_metric="logloss", random_state=RANDOM_STATE, n_jobs=-1, tree_method="hist")),
        {
            "clf__n_estimators": Integer(100, 500),
            "clf__learning_rate": Real(0.01, 0.3, prior="log-uniform"),
            "clf__max_depth": Integer(3, 10),
            "clf__subsample": Real(0.7, 1.00),
            "clf__colsample_bytree": Real(0.70, 1.00)
        }
    ),
    "LinearSVC+Calibrated": (  # linear SVM with probability via calibration
        make_pipe(CalibratedClassifierCV(estimator=LinearSVC(random_state=RANDOM_STATE, max_iter=5000), cv=5)),
        {
            "clf__estimator__C": Real(0.001, 100.000, prior="log-uniform")
        }
    ),
    "MLP": (  # shallow neural net
        make_pipe(MLPClassifier(max_iter=5000, random_state=RANDOM_STATE, solver="adam")),
        {
            "clf__hidden_layer_sizes": Categorical([64, 100, 128]),
            "clf__alpha": Real(0.00001, 0.01, prior="log-uniform"),
            "clf__learning_rate_init": Real(0.0001, 0.01, prior="log-uniform"),
            "clf__activation": Categorical(["relu", "tanh"])
        }
    )
}

# 11) Run: tune -> CV mean±std -> test with ROC AUC + chosen threshold
results = defaultdict(dict)  # hold CV metrics and timing per model
best_experts = {}  # holds best fitted pipelines per model

for name, (pipe, space) in models.items():  # loop over model registry
    print(f"\n🔍 Tuning {name}...")  # log current model
    search, secs = fit_search(pipe, space, X_train, y_train, n_iter=20)  # Bayes search (20 iterations)
    best = search.best_estimator_  # best pipeline (preprocess + SMOTE + classifier)
    best_experts[name] = best 

    # Cross-validated metrics (mean ± std on train via 5 fold CV)
    summary = cv_metrics_formatted(best, X_train, y_train, cv)  # compute CV metrics
    for metric, val in summary.items():  # record each metric
        results[name][f"{metric} (CV)"] = val  # e.g., "roc_auc (CV)": "0.68 ± 0.01"

    # Test evaluation (WARNING: threshold chosen on test
    y_proba = best.predict_proba(X_test)[:, 1]  # test scores (positive-class probability)
    thr = choose_threshold(y_test, y_proba)  # choose F1-maximising threshold (uses test labels)
    y_pred = (y_proba >= thr).astype(int)  #  scores via chosen threshold

    roc = roc_auc_score(y_test, y_proba)  #  ROC-AUC on test
    cm = confusion_matrix(y_test, y_pred)  # confusion matrix at the chosen threshold

    # Print a compact report for this model
    print(f"\n⏱️ {name}: {secs:.2f}s | best_params= {search.best_params_}")  # timing + best hyperparameters
    print(f"ROC AUC (test): {roc:.4f}")  # test ROC-AUC
    print(f"Chosen threshold: {thr:.4f}")  # threshold used for binarisation
    print("\nClassification Report (test set):")  # detailed per-class metrics
    print(classification_report(y_test, y_pred, digits=4))  # precision/recall/F1
    print("Confusion matrix:\n", cm)  # TN/FP/FN/TP




###
# === Summary DataFrame ===
summary = pd.DataFrame(results).T.sort_values("roc_auc (CV)", ascending=False)
print("\n📋 Summary:\n", summary)

###

def split_mean_std(value: str):
    text = str(value).replace("+/-", "±")
    mean_str, std_str = [p.strip() for p in text.split("±")]
    return float(mean_str), float(std_str)

def add_mean_std_columns(df, source_col, out_prefix):
    means, stds = zip(*(split_mean_std(v) for v in df[source_col].values))
    out = df.copy()
    out[f"{out_prefix}_mean"] = means
    out[f"{out_prefix}_std"]  = stds
    return out

def plot_coloured_bar(summary_df, mean_col, std_col, title, y_label):
    rename_map = {"LinearSVC+Calibrated": "LinearSVM"}
    summary_df = summary_df.rename(index=rename_map)

    model_names  = list(summary_df.index)
    metric_means = summary_df[mean_col].to_numpy()
    metric_stds  = summary_df[std_col].to_numpy()
    n = len(model_names)

    base_map = "tab10"
    cmap = plt.colormaps.get_cmap(base_map).resampled(n)
    colours = cmap(np.arange(n))

    fig, ax = plt.subplots(figsize=(10, 4))
    x = np.arange(n)
    bars = ax.bar(x, metric_means, yerr=metric_stds, capsize=4,
                  color=colours, edgecolor="black", linewidth=0.5)

    ax.set_xticks(x, model_names, rotation=30, ha="right")
    ax.set_ylabel(y_label)
    ax.set_title(title)
    ax.set_ylim(0.0, 1.0)                            # <-- force 0–1
    ax.set_yticks(np.linspace(0.0, 1.0, 11))         # 0.0, 0.1, ..., 1.0
    ax.grid(axis="y", linestyle="--", alpha=0.3)

    # Value labels: mean ± std
    for rect, mean, std in zip(bars, metric_means, metric_stds):
        ax.text(rect.get_x() + rect.get_width()/2,
                rect.get_height() + 0.008,          # little padding above bar
                f"{mean:.3f} ± {std:.3f}",
                ha="center", va="bottom", fontsize=9)

    fig.tight_layout()
    plt.show()

# Build numeric columns and plot
summary_num = add_mean_std_columns(summary, "roc_auc (CV)", "roc_auc_cv")
summary_num = summary_num.sort_values("roc_auc_cv_mean", ascending=False)

plot_coloured_bar(
    summary_num,
    mean_col="roc_auc_cv_mean",
    std_col="roc_auc_cv_std",
    title="Experiment 1: Baseline ROC-AUC (CV mean ± std)",
    y_label="ROC-AUC (CV)",
)

# ==========================
# 
# Experiment 2 
#
# ==========================

def pipe_for_cols(best_pipe, cols):
    """Rebuilds your pipeline with the same hyperparameters but a different feature list."""
    clf_frozen = clone(best_pipe.named_steps["clf"])  
    prep = ColumnTransformer([("num", num_pipe, cols)], remainder="drop")  # reuse your num_pipe
    return ImbPipeline(steps=[
        ("prep", prep),
        ("smote", SMOTE(random_state=RANDOM_STATE)),
        ("clf", clf_frozen)
    ])

rows = []
for model_name, best_pipe in best_experts.items():
    # 1) Baseline: ALL features
    pipe_all = pipe_for_cols(best_pipe, user_features)
    scores_all = cross_validate(pipe_all, X_train, y_train, scoring=scoring, cv=cv, n_jobs=-1)
    base_mean = float(np.mean(scores_all["test_roc_auc"]))
    base_std  = float(np.std(scores_all["test_roc_auc"]))

    rows.append({
        "model": model_name, "feature_set": "All", "k": len(user_features),
        "roc_auc_mean": base_mean, "roc_auc_std": base_std,
        "f1_mean": float(np.mean(scores_all["test_f1"])), "f1_std": float(np.std(scores_all["test_f1"])),
        "recall_mean": float(np.mean(scores_all["test_recall"])), "recall_std": float(np.std(scores_all["test_recall"])),
        "delta_vs_all": 0.0

    
    })

    # 2) LOFO: drop each feature once
    for f in user_features:
        cols = [c for c in user_features if c != f]
        pipe_drop = pipe_for_cols(best_pipe, cols)
        scores = cross_validate(pipe_drop, X_train, y_train, scoring=scoring, cv=cv, n_jobs=-1)
        mean_auc = float(np.mean(scores["test_roc_auc"]))
        std_auc  = float(np.std(scores["test_roc_auc"]))

        rows.append({
            "model": model_name, "feature_set": f"All_minus_{f}", "k": len(cols),
            "roc_auc_mean": mean_auc, "roc_auc_std": std_auc,
            "f1_mean": float(np.mean(scores["test_f1"])), "f1_std": float(np.std(scores["test_f1"])),
            "recall_mean": float(np.mean(scores["test_recall"])), "recall_std": float(np.std(scores["test_recall"])),
            "delta_vs_all": mean_auc - base_mean

        
        })

exp2a_lofo = pd.DataFrame(rows).sort_values(["model","feature_set"])
pd.set_option("display.float_format", lambda v: f"{v:.4f}")
print(exp2a_lofo)


summary_rows = []
for m, g in exp2a_lofo.groupby("model"):
    top_hurt = g.loc[g["feature_set"]!="All"].nsmallest(3, "delta_vs_all")[["feature_set","delta_vs_all"]]
    top_help = g.loc[g["feature_set"]!="All"].nlargest(3, "delta_vs_all")[["feature_set","delta_vs_all"]]
    summary_rows.append((m, "most_harmful_to_drop", list(top_hurt.itertuples(index=False))))
    summary_rows.append((m, "drop_helped",          list(top_help.itertuples(index=False))))
pd.DataFrame(summary_rows, columns=["model","category","features_delta"])

avg_delta = (
    exp2a_lofo[exp2a_lofo["feature_set"]!="All"]
    .assign(feature=exp2a_lofo["feature_set"].str.replace("All_minus_","", regex=False))
    .groupby("feature")["delta_vs_all"].agg(["mean","std","count"]).sort_values("mean")
)
print(avg_delta)

# Flatten the 'summary_rows' list-of-tuples into a  DataFrame
flat = []
for model, category, feat_list in summary_rows:
    for item in feat_list:
        # item is like Pandas row tuple: (feature_set, delta)
        feature_set, delta = item
        feature = feature_set.replace("All_minus_", "")
        flat.append({"model": model, "category": category, "feature": feature, "delta_vs_all": float(delta)})

summary_full = pd.DataFrame(flat).sort_values(["model","category","delta_vs_all"])
pd.set_option("display.float_format", lambda v: f"{v:.3f}")
print(summary_full)


summary_full["delta_vs_all"] = pd.to_numeric(summary_full["delta_vs_all"], errors="coerce")
summary_full = summary_full.dropna(subset=["delta_vs_all"])
summary_full = summary_full[(summary_full["delta_vs_all"] >= -1.0) & (summary_full["delta_vs_all"] <= 1.0)]


# Heatmap of Δ AUC (features x models)
lofo_matrix = (
    exp2a_lofo[exp2a_lofo["feature_set"]!="All"]
      .assign(feature=lambda d: d["feature_set"].str.replace("All_minus_","", regex=False))
      .pivot_table(index="feature", columns="model", values="delta_vs_all", aggfunc="mean")
      .reindex(user_features)  # keep your original feature order
)

plt.figure(figsize=(9, 4))
plt.imshow(lofo_matrix.values, aspect="auto")
plt.xticks(range(lofo_matrix.shape[1]), lofo_matrix.columns, rotation=30, ha="right")
plt.yticks(range(lofo_matrix.shape[0]), lofo_matrix.index)
plt.colorbar(label="Δ ROC-AUC vs All")
plt.title("Exp 2A: LOFO — Δ ROC-AUC (mean across CV)")
plt.tight_layout(); plt.show()

# Average Δ across models (single bar plot)
avg_delta_by_feature = lofo_matrix.mean(axis=1)
plt.figure(figsize=(8, 3))
plt.bar(range(len(avg_delta_by_feature)), avg_delta_by_feature.values)
plt.xticks(range(len(avg_delta_by_feature)), avg_delta_by_feature.index, rotation=30, ha="right")
plt.axhline(0, linestyle="--")
plt.ylabel("Average Δ ROC-AUC (vs All)")
plt.title("Exp 2A: Average Δ AUC across models")
plt.tight_layout(); plt.show()


for model_name in lofo_matrix.columns:
    deltas = lofo_matrix[model_name].dropna()
    x_positions = range(len(deltas))
    plt.figure(figsize=(8, 3))
    plt.bar(x_positions, deltas.values)
    plt.xticks(x_positions, deltas.index, rotation=30, ha="right")
    plt.axhline(0, linestyle="--")
    plt.ylabel("Δ ROC-AUC (vs All)")
    plt.title(f"Exp 2A: LOFO — {model_name}")
    plt.tight_layout(); plt.show()


cat_features = [
    "occupancy_status","property_type","loan_purpose","channel",
    "super_conforming_flag","interest_only_indicator","prepayment_penalty_flag",
    "property_state" 
]
extended_cols = user_features + cat_features


X_ext = df[extended_cols].copy()
y_ext = df["default_flag"].astype(int).copy()
X_train_ext, X_test_ext, y_train_ext, y_test_ext = train_test_split(
    X_ext, y_ext, test_size=0.2, stratify=y_ext, random_state=RANDOM_STATE
)

cat_pipe = SkPipeline(steps=[
    ("imp", SimpleImputer(strategy="most_frequent")),
    ("oh", OneHotEncoder(handle_unknown="ignore"))
])

def pipe_num_cat(best_pipe, num_cols, cat_cols):
    prep = ColumnTransformer([
        ("num", num_pipe, num_cols), 
        ("cat", cat_pipe, cat_cols)
    ], remainder="drop")
    clf = clone(best_pipe.named_steps["clf"])
    return ImbPipeline([("prep", prep), ("smote", SMOTE(random_state=RANDOM_STATE)), ("clf", clf)])

rows = []
for model_name, best_pipe in best_experts.items():
    # Base10
    base_pipe = pipe_num_cat(best_pipe, user_features, [])
    base_scores = cross_validate(base_pipe, X_train_ext, y_train_ext, scoring=scoring, cv=cv, n_jobs=-1)
    base_mean = np.mean(base_scores["test_roc_auc"]); base_std = np.std(base_scores["test_roc_auc"])
    rows.append({"model": model_name, "set": "Base10", "roc_auc_mean": base_mean, "roc_auc_std": base_std})

    # Base10 + Cats
    ext_pipe = pipe_num_cat(best_pipe, user_features, cat_features)
    ext_scores = cross_validate(ext_pipe, X_train_ext, y_train_ext, scoring=scoring, cv=cv, n_jobs=-1)
    ext_mean = np.mean(ext_scores["test_roc_auc"]); ext_std = np.std(ext_scores["test_roc_auc"])
    rows.append({"model": model_name, "set": "Base10+Cats", "roc_auc_mean": ext_mean, "roc_auc_std": ext_std,
                 "delta": ext_mean - base_mean})

exp2b = pd.DataFrame(rows).sort_values(["model","set"])
print(exp2b)


def _disp_name(model):
    if model == "LinearSVC+Calibrated":
        return "LinearSVM"
    return model

def plot_exp2b_all_models_simple(exp2b):
    df = exp2b.copy()
    df["model"] = df["model"].astype(str).str.strip()
    df["set"]   = df["set"].astype(str).str.strip()

    set_order   = ["Base10", "Base10+Cats"]
    df = df[df["set"].isin(set_order)]
    if df.empty:
        print("Nothing to plot: exp2b has no rows with sets Base10 or Base10+Cats.")
        print("Available sets:", exp2b["set"].unique())
        return

    
    model_order = [m for m in df["model"].unique()]
    df["set_order"] = df["set"].map({s:i for i,s in enumerate(set_order)})
    df = df.sort_values(["model","set_order"])

    # labels, positions, colours
    labels = [f"{_disp_name(r.model)} {r.set}" for r in df.itertuples()]
    means  = df["roc_auc_mean"].to_numpy().astype(float)
    stds   = df["roc_auc_std"].to_numpy().astype(float)

    cmap = plt.get_cmap("tab10")
    color_map = {m: cmap(i % 10) for i, m in enumerate(model_order)}
    colors = [color_map[m] for m in df["model"]]

    x = np.arange(len(labels))

    # --- draw ---
    fig, ax = plt.subplots(figsize=(max(12, 0.7*len(labels)), 5))
    bars = ax.bar(x, means, yerr=stds, capsize=4,
                  color=colors, edgecolor="black", linewidth=0.5)

    ax.set_xticks(x, labels, rotation=30, ha="right")
    
    top = float((means + stds).max() + 0.03)
    ax.set_ylim(0.0, min(1.0, top))
    ax.set_yticks(np.linspace(0.0, 1.0, 11))
    ax.set_ylabel("ROC-AUC (CV)")
    ax.set_title("Exp 2b: Base10 vs Base10+Cats — All Models")
    ax.grid(axis="y", linestyle="--", alpha=0.3)

    
    for rect, mean, std in zip(bars, means, stds):
        ax.annotate(f"{mean:.3f}±{std:.3f}",
                    xy=(rect.get_x() + rect.get_width()/2, rect.get_height() + std),
                    xytext=(0, 6), textcoords="offset points",
                    ha="center", va="bottom",
                    rotation=90, fontsize=9, clip_on=False)

    fig.tight_layout()
    plt.show()


plot_exp2b_all_models_simple(exp2b)


# ==========================
# 
# Experiment 3 
#
# ==========================

# ----- config  -----
RANDOM_STATE = 29
cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=RANDOM_STATE)

# ----- models & tests  -----
models = {
    "LR":  LogisticRegression(C=0.009604215688369763, solver="saga", max_iter=500, random_state=RANDOM_STATE),
    "DT":  DecisionTreeClassifier(max_depth=3, min_samples_leaf=9, min_samples_split=2, random_state=RANDOM_STATE),
    "RF":  RandomForestClassifier(max_depth=5, min_samples_split=20, n_estimators=400, random_state=RANDOM_STATE, n_jobs=-1),
    "SVM": CalibratedClassifierCV(estimator=LinearSVC(C=0.001, max_iter=5000, random_state=RANDOM_STATE),cv=5),
    "XGB": XGBClassifier(colsample_bytree=0.8000686261947085, learning_rate=0.010644672992077926, max_depth=3, n_estimators=341, subsample=0.9275534913741973, tree_method="hist", eval_metric="logloss", random_state=RANDOM_STATE, n_jobs=-1),
    "MLP": MLPClassifier(solver="adam", max_iter=5000, random_state=RANDOM_STATE, activation="tanh", alpha=0.00728922087100724, hidden_layer_sizes=64, learning_rate_init=0.0001431133506494306),
}

tests = {
    "Unbalanced": None,
    "Oversampling": SMOTE(sampling_strategy=1.0, random_state=RANDOM_STATE),
    "Downsampling": RandomUnderSampler(sampling_strategy=1.0, random_state=RANDOM_STATE),
}

def make_pipe(sampler, clf):
    steps = [("imputer", SimpleImputer(strategy="median")),
             ("scaler", StandardScaler())]
    if sampler is not None:
        steps.append(("sampler", sampler))  # resample on train folds only
    steps.append(("clf", clf))
    return ImbPipeline(steps)

# ----- compute metrics -----
rows = []
for test_name, sampler in tests.items():
    for model_name, clf in models.items():
        pipe = make_pipe(sampler, clf)

        # ROC-AUC via CV (mean±std) + wall time
        t0 = time.perf_counter()
        cv_out = cross_validate(pipe, X, y,
                                scoring={"roc_auc": "roc_auc"},
                                cv=cv, n_jobs=-1, return_train_score=False)
        wall = time.perf_counter() - t0
        roc_mean = float(np.mean(cv_out["test_roc_auc"]))
        roc_std  = float(np.std(cv_out["test_roc_auc"]))

        # per-fold confusion matrices (for mean±std of TN/FP/FN/TP)
        tn_list, fp_list, fn_list, tp_list = [], [], [], []
        for tr_idx, te_idx in cv.split(X, y):
            X_tr = X.iloc[tr_idx] if hasattr(X, "iloc") else X[tr_idx]
            X_te = X.iloc[te_idx] if hasattr(X, "iloc") else X[te_idx]
            y_tr = y.iloc[tr_idx] if hasattr(y, "iloc") else y[tr_idx]
            y_te = y.iloc[te_idx] if hasattr(y, "iloc") else y[te_idx]

            pipe_fold = make_pipe(clone(sampler) if sampler is not None else None, clone(clf)).fit(X_tr, y_tr)
            y_pred = pipe_fold.predict(X_te)
            tn, fp, fn, tp = confusion_matrix(y_te, y_pred, labels=[0, 1]).ravel()
            tn_list.append(tn); fp_list.append(fp); fn_list.append(fn); tp_list.append(tp)

        tn_mean, tn_std = np.mean(tn_list), np.std(tn_list)
        fp_mean, fp_std = np.mean(fp_list), np.std(fp_list)
        fn_mean, fn_std = np.mean(fn_list), np.std(fn_list)
        tp_mean, tp_std = np.mean(tp_list), np.std(tp_list)

        rows.append({
            "test": test_name,
            "model_group": model_name,
            "time_taken": float(wall),
            "roc_auc_mean": roc_mean, "roc_auc_std": roc_std,
            "tn_mean": tn_mean, "tn_std": tn_std,
            "fp_mean": fp_mean, "fp_std": fp_std,
            "fn_mean": fn_mean, "fn_std": fn_std,
            "tp_mean": tp_mean, "tp_std": tp_std,
            # pretty strings
            "tn": f"{int(round(tn_mean))}±{int(round(tn_std))}",
            "fp": f"{int(round(fp_mean))}±{int(round(fp_std))}",
            "fn": f"{int(round(fn_mean))}±{int(round(fn_std))}",
            "tp": f"{int(round(tp_mean))}±{int(round(tp_std))}",
            "roc_auc_str": f"{roc_mean:.3f} ± {roc_std:.3f}",
            # aliases you asked for previously
            "mean": roc_mean, "deviation": roc_std,
        })

exp3_cv = (pd.DataFrame(rows).sort_values(["test", "model_group"]).reset_index(drop=True))

# ----- tables -----
print("\nConfusion matrices (mean±std over folds):")
print(exp3_cv[["test","model_group","tn","fp","fn","tp"]])

print("\nROC-AUC (mean±std) by test:")
print(exp3_cv.pivot(index="test", columns="model_group", values="roc_auc_str"))

# ----- single ROC-AUC grouped bar  -----
tests_order  = ["Unbalanced", "Oversampling", "Downsampling"]
models_order = ["DT", "SVM", "LR", "MLP", "RF", "XGB"]  # change order if you like
colors       = {"Unbalanced": "red", "Oversampling": "green", "Downsampling": "blue"}

mean_tbl = (exp3_cv.pivot(index="model_group", columns="test", values="roc_auc_mean").reindex(index=models_order, columns=tests_order))
std_tbl  = (exp3_cv.pivot(index="model_group", columns="test", values="roc_auc_std").reindex(index=models_order, columns=tests_order))

x = np.arange(len(models_order))
width = 0.8 / len(tests_order)

plt.close('all')  
fig, ax = plt.subplots(figsize=(10, 4.8))
for j, t in enumerate(tests_order):
    xpos = x - 0.4 + width/2 + j*width
    means = mean_tbl[t].values
    errs  = std_tbl[t].values
    bars = ax.bar(xpos, means, yerr=errs, capsize=4, width=width, color=colors[t], label=t)
    for rect, val in zip(bars, means):
        ax.text(rect.get_x() + rect.get_width()/2, val + 0.02, f"{val:.3f}", ha="center", va="bottom", fontsize=9)

ax.set_xticks(x)
ax.set_xticklabels(models_order)
ax.set_ylabel("ROC-AUC (CV)")
ax.set_title("Exp 3: Class Imbalance Strategies — All Models")
ax.legend(loc="center left", bbox_to_anchor=(1.02, 0.5), frameon=True)
ax.set_ylim(0.0, min(1.0, float(mean_tbl.values.max() + std_tbl.values.max() + 0.1)))
ax.grid(axis="y", linestyle="--", alpha=0.3)
plt.subplots_adjust(right=0.82)
plt.tight_layout()
plt.show()
plt.close(fig)

# ----- dataset size snapshot (one stratified 80/20 split, then show class balance) -----
X_tr, X_te, y_tr, y_te = train_test_split(X, y, test_size=0.20, stratify=y, random_state=RANDOM_STATE)
imp = SimpleImputer(strategy="median")
X_tr_imp = imp.fit_transform(X_tr)
X_te_imp = imp.transform(X_te)

print("\n✅ Dataset Split Complete:")
print("X_test_clean:", X_te.shape)
print("y_test_clean distribution:\n", pd.Series(y_te).value_counts())

alias = {"Unbalanced": "unbal", "Oversampling": "over", "Downsampling": "down"}
for test_name, sampler in tests.items():
    tag = alias[test_name]
    if sampler is None:
        X_res, y_res = X_tr_imp, y_tr
    else:
        X_res, y_res = clone(sampler).fit_resample(X_tr_imp, y_tr)
    print(f"\n— {test_name} —")
    print(f"X_train_{tag}:", X_res.shape)
    print(f"y_train_{tag} distribution:\n", pd.Series(y_res).value_counts())


# ==========================
# 
# Experiment 4 
#
# ==========================

RANDOM_STATE = 29
cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=RANDOM_STATE)

def make_pipe(clf, sampler=None):
    steps = [("imputer", SimpleImputer(strategy="median")),
             ("scaler", StandardScaler())]
    if sampler is not None:
        steps.append(("sampler", sampler))
    steps.append(("clf", clf))
    return ImbPipeline(steps)

def find_threshold(y_true, scores, method="youden"):
    """
    Choose a probability threshold on training data only.
    method="youden": maximise (TPR - FPR)
    method="f1":     maximise F1 on the PR curve
    """
    y_true = np.asarray(y_true)
    scores = np.asarray(scores)

    if method == "youden":
        fpr, tpr, thr = roc_curve(y_true, scores)
        idx = np.argmax(tpr - fpr)
        return thr[idx]
    elif method == "f1":
        prec, rec, thr = precision_recall_curve(y_true, scores)
        # precision_recall_curve returns thresholds of length n-1
        f1 = (2 * prec[:-1] * rec[:-1]) / (prec[:-1] + rec[:-1] + 1e-12)
        return thr[np.argmax(f1)]
    else:
        raise ValueError("method must be 'youden' or 'f1'")

# ----- tuned experts (same as Exp3; no sampler for Exp4 ) -----
experts = {
    "LR":  LogisticRegression(C=0.009604215688369763, solver="saga", max_iter=500,  random_state=RANDOM_STATE),
    "RF":  RandomForestClassifier(max_depth=5, min_samples_split=20, n_estimators=400, random_state=RANDOM_STATE, n_jobs=-1),
    "XGB": XGBClassifier(colsample_bytree=0.8000686261947085, learning_rate=0.010644672992077926,
                         max_depth=3, n_estimators=341, subsample=0.9275534913741973,
                         tree_method="hist", eval_metric="logloss", random_state=RANDOM_STATE, n_jobs=-1),
    "MLP": MLPClassifier(solver="adam", max_iter=5000, random_state=RANDOM_STATE,
                         activation="tanh", alpha=0.00728922087100724,
                         hidden_layer_sizes=64, learning_rate_init=0.0001431133506494306),
    "SVM": CalibratedClassifierCV(
                estimator=LinearSVC(C=0.001, max_iter=5000, random_state=RANDOM_STATE),
                cv=5, method="sigmoid")
}

#get probability/score consistently
def predict_proba_1d(pipe, X):
    if hasattr(pipe, "predict_proba"):
        return pipe.predict_proba(X)[:, 1]
    else:
        # for decision_function-based models (not needed here since SVM is calibrated)
        return pipe.decision_function(X)

# ----- Exp4: outer CV -----
rows = []
cms_moe = []  
t0_all = time.perf_counter()

for fold_idx, (tr_idx, te_idx) in enumerate(cv.split(X, y), 1):
    X_tr = X.iloc[tr_idx] if hasattr(X, "iloc") else X[tr_idx]
    X_te = X.iloc[te_idx] if hasattr(X, "iloc") else X[te_idx]
    y_tr = y.iloc[tr_idx] if hasattr(y, "iloc") else y[tr_idx]
    y_te = y.iloc[te_idx] if hasattr(y, "iloc") else y[te_idx]

    # --- Step A: build OOF matrix on training fold via inner split ---
    inner = StratifiedKFold(n_splits=5, shuffle=True, random_state=RANDOM_STATE)
    oof = np.zeros((len(y_tr), len(experts)), dtype=float)
    col_names = list(experts.keys())

    for e_idx, (name, clf) in enumerate(experts.items()):
        oof_fold = np.zeros(len(y_tr), dtype=float)
        for in_tr, in_va in inner.split(X_tr, y_tr):
            X_a, X_b = X_tr.iloc[in_tr], X_tr.iloc[in_va]
            y_a, y_b = y_tr.iloc[in_tr], y_tr.iloc[in_va]
            pipe = make_pipe(clone(clf))  # no sampler in Exp4 v1
            pipe.fit(X_a, y_a)
            oof_fold[in_va] = predict_proba_1d(pipe, X_b)
        oof[:, e_idx] = oof_fold

    # --- Step B: train gate on OOF ---
    gate = LogisticRegression(max_iter=2000, solver="lbfgs", random_state=RANDOM_STATE)
    gate.fit(oof, y_tr)

    # --- Step C: refit experts on full training fold and predict test fold ---
    test_stack = np.zeros((len(y_te), len(experts)), dtype=float)
    single_scores = {}  # for baselines
    for e_idx, (name, clf) in enumerate(experts.items()):
        pipe_full = make_pipe(clone(clf))
        pipe_full.fit(X_tr, y_tr)
        proba_te = predict_proba_1d(pipe_full, X_te)
        test_stack[:, e_idx] = proba_te
        single_scores[name] = roc_auc_score(y_te, proba_te)

    # --- MoE predictions + baselines ---
    moe_probs = gate.predict_proba(test_stack)[:, 1]
    avg_probs = test_stack.mean(axis=1)

    # collect AUCs
    auc_moe = roc_auc_score(y_te, moe_probs)
    auc_avg = roc_auc_score(y_te, avg_probs)
    best_single_name = max(single_scores, key=single_scores.get)
    auc_best_single = single_scores[best_single_name]

    rows.append({
        "fold": fold_idx,
        "AUC_MOE": auc_moe,
        "AUC_AVG": auc_avg,
        "AUC_best_single": auc_best_single,
        "best_single_model": best_single_name
    })

    # learn threshold on OUTER-TRAIN using the gate's OOF predictions
    train_gate_probs = gate.predict_proba(oof)[:, 1]
    thr_moe = find_threshold(y_tr, train_gate_probs, method="youden")  # or "f1"

    # apply to OUTER-TEST
    y_pred_moe = (moe_probs >= thr_moe).astype(int)
    tn, fp, fn, tp = confusion_matrix(y_te, y_pred_moe, labels=[0, 1]).ravel()
    cms_moe.append([tn, fp, fn, tp])

t_all = time.perf_counter() - t0_all
exp4 = pd.DataFrame(rows)

# ---- summary ----
def ms(x): return f"{x.mean():.3f} ± {x.std():.3f}"
print("\nExp 4 — Mixture of Experts (outer 5-fold):")
print("MOE (gate LR):", ms(exp4["AUC_MOE"]))
print("Average probs :", ms(exp4["AUC_AVG"]))
print("Best single   :", ms(exp4["AUC_best_single"]))
print("Time taken    :", f"{t_all:.1f}s")
print("\nBest single per fold:\n", exp4[["fold","best_single_model","AUC_best_single"]])

# MOE confusion matrix mean ± std (two-line format) ----
cms_moe = np.array(cms_moe)  # shape (k, 4)
tn_mean, fp_mean, fn_mean, tp_mean = cms_moe.mean(axis=0)
tn_std,  fp_std,  fn_std,  tp_std  = cms_moe.std(axis=0)

cm_str = (
    f"[[{int(round(tn_mean))} ± {int(round(tn_std))}  {int(round(fp_mean))} ± {int(round(fp_std))}]\n"
    f" [{int(round(fn_mean))} ± {int(round(fn_std))}  {int(round(tp_mean))} ± {int(round(tp_std))}]]"
)
print("\nMOE confusion matrix (mean ± std over folds):")
print(cm_str)

cms_moe = np.array(cms_moe)
tn_m, fp_m, fn_m, tp_m = cms_moe.mean(axis=0)
prec = tp_m / (tp_m + fp_m + 1e-12)
rec  = tp_m / (tp_m + fn_m + 1e-12)
f1   = 2*prec*rec / (prec + rec + 1e-12)
print(f"\nMOE @ train-picked threshold — Precision ≈ {prec:.3f}, Recall ≈ {rec:.3f}, F1 ≈ {f1:.3f}\n")


# 1) AUC summary (mean ± std)
def ms(x): return f"{np.mean(x):.3f} ± {np.std(x):.3f}"
moe  = exp4["AUC_MOE"].values
avg  = exp4["AUC_AVG"].values
best = exp4["AUC_best_single"].values

summary = pd.DataFrame({
    "Metric": ["ROC-AUC"],
    "MOE (gate LR)": [ms(moe)],
    "Average probs": [ms(avg)],
    "Best single"   : [ms(best)],
})
print(summary.to_string(index=False))

# 2) Paired tests (per-fold)
delta_moe_best = (moe - best)
ttest = stats.ttest_rel(moe, best)
wil   = stats.wilcoxon(moe, best)

tests_tbl = pd.DataFrame({
    "Test": ["Paired t-test", "Wilcoxon signed-rank"],
    "Statistic": [float(ttest.statistic), float(wil.statistic)],
    "p-value": [float(ttest.pvalue), float(wil.pvalue)],
    "H0 (no diff) rejected @ 0.1?": [ttest.pvalue < 0.1, wil.pvalue < 0.1]
})
print("\nPaired tests (MOE vs Best single):")
print(tests_tbl.to_string(index=False))

print("\nPer-fold Δ(MOE − Best):", np.round(delta_moe_best, 4))

# 3) bar chart of mean AUCs with ± std 
labels = ["MOE", "Average", "Best single"]
means  = [np.mean(moe), np.mean(avg), np.mean(best)]
stds   = [np.std(moe),  np.std(avg),  np.std(best)]

plt.close('all')
fig, ax = plt.subplots(figsize=(6, 4))
x = np.arange(len(labels))
bars = ax.bar(x, means, yerr=stds, capsize=5)

# value labels
for rect, val, s in zip(bars, means, stds):
    ax.text(rect.get_x() + rect.get_width()/2, val + 0.01, f"{val:.3f}" +"±"+ f"{s:.3f}",
            ha="center", va="bottom", fontsize=10)

ax.set_xticks(x)
ax.set_xticklabels(labels)
ax.set_ylabel("ROC-AUC (mean ± std)")
ax.set_title("Experiment 4 — Mixture of Experts (outer 5-fold)")
ax.set_ylim(0, min(1.0, max(means) + max(stds) + 0.07))
ax.grid(axis="y", linestyle="--", alpha=0.3)
plt.tight_layout()
plt.show()

